<h1>API 1</h1>

In [1]:
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files

In [2]:
API = f"https://openlibrary.org/search.json"
search_list = ['love','detective','marvel','war','history','science','fiction','horror','mystery','adventure']
data_buku = []
for search in search_list:
    response = requests.get(API, params={'q': search})
    data = response.json()
    buku = data.get('docs', [])
    for b in buku[:10]:
        judul = b.get('title', 'N/A')
        author = ', '.join(b.get('author_name', ['N/A']))
        tahun_publikasi = b.get('first_publish_year', 'N/A')
        data_buku.append({
            'pencarian': search,
            'title': judul,
            'author': author,
            'first_publish_year': tahun_publikasi})

df = pd.DataFrame(data_buku)
df.to_csv('buku.csv', index=False)

output_file = 'buku.csv'
files.download(output_file)
df







<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,pencarian,title,author,first_publish_year
0,love,Love and Freindship,Jane Austen,1995
1,love,Women in Love,D. H. Lawrence,1877
2,love,The Art of Loving,Erich Fromm,1956
3,love,Guess How Much I Love You,"Sam McBratney, Anita Jeram",1994
4,love,Motivation Noteook,Raymond Gething,2020
...,...,...,...,...
95,adventure,She,H. Rider Haggard,1886
96,adventure,Alice's Adventures in Wonderland,Lewis Carroll,1865
97,adventure,Alice's Adventures in Wonderland / Through the...,Lewis Carroll,1889
98,adventure,Le avventure di Pinocchio,Carlo Collodi,1883


<h1>API 2</h1>

In [3]:
import requests
import pandas as pd

URL = "https://api.spacexdata.com/v5/launches/past"

res = requests.get(URL, timeout=20)
print("Status Code:", res.status_code)
if res.status_code != 200:
    raise Exception("Gagal mengakses API SpaceX")

data = res.json()

rows = []
for it in data[-10:]:
    rows.append({
        "nama_misi": it.get("name"),
        "tanggal": it.get("date_utc"),
        "id_roket": it.get("rocket"),
        "detail": it.get("details"),
        "link_webcast": it.get("links", {}).get("webcast")
    })

df = pd.DataFrame(rows)
print(f"Total item: {len(df)}")
df.to_csv("launches_spacex.csv", index=False)

output_file = 'launches_spacex.csv'
files.download(output_file)
df

Status Code: 200
Total item: 10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,nama_misi,tanggal,id_roket,detail,link_webcast
0,Starlink 4-26 (v1.5),2022-08-09T22:57:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/ck5z0uMGz8s
1,Starlink 3-3 (v1.5),2022-08-12T21:30:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/SU5FbiCbjic
2,Starlink 4-27 (v1.5),2022-08-19T19:24:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/M018DAaNd_E
3,Starlink 4-23 (v1.5),2022-08-28T02:22:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/07RGJ04HRns
4,Starlink 3-4 (v1.5),2022-08-31T05:40:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/zSJWK_pmXVw
5,Starlink 4-20 (v1.5) & Sherpa LTC-2/Varuna-TDM,2022-09-05T02:09:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/NONM-xsKMSs
6,Starlink 4-2 (v1.5) & Blue Walker 3,2022-09-11T01:10:00.000Z,5e9d0d95eda69973a809d1ec,None,None
7,Starlink 4-34 (v1.5),2022-09-17T01:05:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/ZlQHF_yBkMQ
8,Starlink 4-35 (v1.5),2022-09-24T23:30:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/VVu2bSJJhgI
9,Crew-5,2022-10-05T16:00:00.000Z,5e9d0d95eda69973a809d1ec,None,https://youtu.be/5EwW8ZkArL4


<h1>HTTP 2</h1>

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    text = re.sub(r'\[\s*\w+\s*\]', '', text)
    parts = re.split(r'(?<=[\.\?\!])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def scrape_title_and_top_sentences(url, max_sentences_per_paragraph=10):
    print(f"Mengambil: {url}")
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    title_tag = soup.find("h1", id="firstHeading")
    page_title = title_tag.get_text(strip=True) if title_tag else ""

    content_div = soup.find("div", class_="mw-page-container")
    paragraphs = content_div.find_all("p") if content_div else []

    rows = []
    for idx, p in enumerate(paragraphs):
        raw = p.get_text(separator=" ", strip=True)
        if not raw:
            continue
        cleaned = re.sub(r'\[\s*\d+\s*(?:,\s*\d+)*\s*\]', '', raw)
        sents = split_sentences(cleaned)
        top_sents = sents[:max_sentences_per_paragraph]
        rows.append({
            "page_title": page_title,
            "paragraph_index": idx,
            "paragraph_text": cleaned,
            "top_sentences": " ".join(top_sents),
            "num_sentences_in_paragraph": len(sents),
        })

    return pd.DataFrame(rows)

link = "https://id.wikipedia.org/wiki/Toyota_Avanza"
df = scrape_title_and_top_sentences(link, max_sentences_per_paragraph=10)

pd.set_option('display.max_colwidth', 200)
df.to_csv("wikipedia_toyota_avanza.csv", index=False)

output_file = 'wikipedia_toyota_avanza.csv'
files.download(output_file)
df

Mengambil: https://id.wikipedia.org/wiki/Toyota_Avanza


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,page_title,paragraph_index,paragraph_text,top_sentences,num_sentences_in_paragraph
0,Toyota Avanza,0,"Toyota Avanza dan Daihatsu Xenia adalah mobil berjenis kendaraan multi-guna (MPV) produk asli Indonesia yang dikembangkan oleh Daihatsu dan dipasarkan oleh Toyota dan Daihatsu, utamanya dijual den...","Toyota Avanza dan Daihatsu Xenia adalah mobil berjenis kendaraan multi-guna (MPV) produk asli Indonesia yang dikembangkan oleh Daihatsu dan dipasarkan oleh Toyota dan Daihatsu, utamanya dijual den...",2
1,Toyota Avanza,1,"Nama ""Avanza"" berasal dari bahasa Italia , yaitu "" Avanzato "" dan bahasa Spanyol , yaitu "" Avanza "", yang berarti peningkatan. Sedangkan nama ""Xenia"" diambil dari bahasa Yunani , yaitu "" Xenia "", ...","Nama ""Avanza"" berasal dari bahasa Italia , yaitu "" Avanzato "" dan bahasa Spanyol , yaitu "" Avanza "", yang berarti peningkatan. Sedangkan nama ""Xenia"" diambil dari bahasa Yunani , yaitu "" Xenia "", ...",3
2,Toyota Avanza,2,"Avanza dan Xenia mulai digagas oleh Toyota, Daihatsu, dan anak perusahaan mereka di Indonesia di saat terjadinya krisis keuangan Asia pada tahun 1997 . Pada waktu itu, harga dari Toyota Kijang yan...","Avanza dan Xenia mulai digagas oleh Toyota, Daihatsu, dan anak perusahaan mereka di Indonesia di saat terjadinya krisis keuangan Asia pada tahun 1997 . Pada waktu itu, harga dari Toyota Kijang yan...",5
3,Toyota Avanza,3,"Toyota juga merujuk proyek ini sebagai proyek U-IMV (Under-IMV), sebuah referensi di mana Innova yang memakai platform IMV berada di segmen di atas Avanza. Tidak seperti Kijang dan penerusnya, Kij...","Toyota juga merujuk proyek ini sebagai proyek U-IMV (Under-IMV), sebuah referensi di mana Innova yang memakai platform IMV berada di segmen di atas Avanza. Tidak seperti Kijang dan penerusnya, Kij...",4
4,Toyota Avanza,4,Generasi kedua memakai platform yang sama dengan generasi pertamanya dan dikembangkan selama 4 tahun. Toyota dan Daihatsu menginvestasikan biaya total Rp 900.000.000.000 untuk proyek ini.,Generasi kedua memakai platform yang sama dengan generasi pertamanya dan dikembangkan selama 4 tahun. Toyota dan Daihatsu menginvestasikan biaya total Rp 900.000.000.000 untuk proyek ini.,2
...,...,...,...,...,...
76,Toyota Avanza,76,Limited,Limited,1
77,Toyota Avanza,77,"Per Oktober 2011, Avanza/Xenia telah terjual sekitar 1 juta unit, termasuk sekitar 113,000 unit ekspor. Di tahun 2012, Avanza/Xenia menjadi MPV paling laris kedua, setelah Minivan Chrysler.","Per Oktober 2011, Avanza/Xenia telah terjual sekitar 1 juta unit, termasuk sekitar 113,000 unit ekspor. Di tahun 2012, Avanza/Xenia menjadi MPV paling laris kedua, setelah Minivan Chrysler.",2
78,Toyota Avanza,78,"Per November 2018, Avanza/Xenia telah terjual 2,750,576 unit secara global. Dalam penjualan ini, sebanyak 1,714,196 unit Avanza terjual di Indonesia, 393,380 Avanza diekspor ke luar negeri, dan se...","Per November 2018, Avanza/Xenia telah terjual 2,750,576 unit secara global. Dalam penjualan ini, sebanyak 1,714,196 unit Avanza terjual di Indonesia, 393,380 Avanza diekspor ke luar negeri, dan se...",2
79,Toyota Avanza,79,"Dari tahun 2006 sampai 2019 dan 2021, Avanza menjadi mobil penumpang terlaris di Indonesia, sebelum diambil-alih oleh Honda Brio Hatchback pada tahun 2020 dan 2022.","Dari tahun 2006 sampai 2019 dan 2021, Avanza menjadi mobil penumpang terlaris di Indonesia, sebelum diambil-alih oleh Honda Brio Hatchback pada tahun 2020 dan 2022.",1


<h1>HTTP 2</h1>

In [5]:
url = "https://www.neliti.com/id/journals"
headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<!--

   ````                                                          /ooo/    :///-                :///-
 :osooss/`                                                       /ooo/   .oooo+    `.-~.      `ooooo
+s+::::/ss`                                                      /ooo/    -:/:.    .oooo`      -:/:.
ss::::::+s-                 .-~.  .-~-~-`         .-:::::-`      /ooo/    .-~.`  .-:oooo-~-~-~-~-~.`
:so/:::+so`                `ooo+:+ooooooo/`     :+ooooooooo/.    /ooo/    oooo/  +ooooooooooooooooo/
 ./osso+o-        -oso/    `oooooo/-:+oooo/   `+ooo/.  `:oooo.   /ooo/    oooo/  -~:oooo:-~-~-~oooo/
        `//-~-.  `sssss.   `oooo/`    +oooo   :ooo+``````+ooo:   /ooo/    oooo/    .oooo`      +ooo/
         -ssssso//-:+/.    `oooo-     /oooo   +oooooooooooooo/   /ooo/    oooo/    .oooo`      +ooo/
         +ssssss:          `oooo-     /oooo   +ooo+-~-~-~-~-~`   /ooo/    oooo/    .oooo`      +ooo/
         /ssssss`          `oooo-     /oooo   -oooo-             /ooo

In [6]:
semua_jurnal = soup.find_all('div', {'class':'result'})
data_jurnal = []

for j in semua_jurnal:
    nama_jurnal = j.find('a').get_text(strip=True)
    deskripsi = j.find('a', {'class':'link'}).get_text(strip=True)
    data_jurnal.append({
        'nama_jurnal': nama_jurnal,
        'instansi': deskripsi
    })

df = pd.DataFrame(data_jurnal)
df.to_csv('jurnal_neliti.csv', index=False)

output_file = 'jurnal_neliti.csv'
files.download(output_file)
df

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,nama_jurnal,instansi
0,Eastern-European Journal of Enterprise Technologies (EEJET),PC TECHNOLOGY CENTER 🇺🇦
1,International Journal of Health Sciences (IJHS),ScienceScholar 🇪🇨
2,JournalNX,Novateur Publication 🇮🇳
3,Jurnal Pendidikan dan Pembelajaran Untan,Universitas Tanjungpura 🇮🇩
4,KnE Social Sciences,Knowledge E 🇦🇪
5,Jurnal Online Mahasiswa Fakultas Ilmu Sosial dan Ilmu Politik Universitas Riau (JOM FISIP UNRI),Universitas Riau 🇮🇩
6,Jurnal Teknik ITS,Institut Teknologi Sepuluh Nopember 🇮🇩
7,Technology Audit and Production Reserves,PC TECHNOLOGY CENTER 🇺🇦
8,Galaxy International Interdisciplinary Research Journal (GIIRJ),Novateur Publication 🇮🇳
9,Jurnal Khatulistiwa Informatika,Bina Sarana Informatika 🇮🇩
